# Comparaison des références TD1

Exécuter les deux cellules pour comparer les requêtes SQLite, le calcul indépendant sur CSV et les montants enregistrés. Les montants sont en centimes HT.

In [1]:
"""Build SQL references and independently check CSV totals; no model API calls."""
import csv
import json
import re
import sqlite3
from datetime import date, datetime
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'reference-td1.csv').exists():
    raise FileNotFoundError('Ouvrir le notebook depuis la racine du projet.')


def read_csv(name):
    with (ROOT / 'data' / name).open(encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def csv_total(question, orders, customers, prices):
    # Parse the original wording independently of the SQL specifications below.
    timestamps = re.findall(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', question)
    period = [datetime.fromisoformat(t) for t in timestamps]
    reference = re.search(r'reference date (\d{4}-\d{2}-\d{2})', question)
    products = re.search(r'products? (.*?) only', question)
    allowed = set(re.findall(r'\b[a-e]\b', products.group(1))) if products else None
    total = 0
    for row in orders:
        placed = datetime.fromisoformat(row['date'])
        if period and not period[0] <= placed < period[1]:
            continue
        if allowed is not None and row['product_id'] not in allowed:
            continue
        if reference or 'at the time of the order' in question:
            on = date.fromisoformat(reference.group(1)) if reference else placed.date()
            birth = customers[row['customer_id']]
            # Birthday interval, independent of SQL's completed-age subtraction.
            if not birth.replace(year=birth.year + 25) <= on < birth.replace(year=birth.year + 26):
                continue
        total += int(row['quantity_product']) * prices[row['product_id']]
    return total


def main():
    questions = json.loads((ROOT / 'questions-td1.json').read_text(encoding='utf-8'))['questions']
    with (ROOT / 'reference-td1.csv').open(encoding='utf-8', newline='') as f:
        references = list(csv.DictReader(f))
    expected_ids = [q['id'] for q in questions]
    if len(references) != 20 or [r['case_id'] for r in references] != expected_ids:
        raise ValueError('Le fichier de reference doit contenir C01 a C20 dans cet ordre.')
    orders = read_csv('orders.csv')
    customers = {r['customer_id']: date.fromisoformat(r['birth_date']) for r in read_csv('customers.csv')}
    prices = {r['product_id']: int(r['price']) for r in read_csv('product.csv')}
    headers = ['Question', 'SQL execute', 'Calcul CSV', 'Reference', 'Ecart SQL/CSV', 'Statut']
    table = []
    def number(value):
        return format(value, ',').replace(',', ' ')
    with sqlite3.connect((ROOT / 'data/boutique.sqlite').as_uri() + '?mode=ro', uri=True) as con:
        con.execute('PRAGMA query_only = 1')
        for question, ref in zip(questions, references):
            cursor = con.execute(ref['sql'])
            result = cursor.fetchall()
            if [d[0] for d in cursor.description] != ['revenue_cents'] or len(result) != 1 or len(result[0]) != 1 or type(result[0][0]) is not int:
                raise ValueError(f"{ref['case_id']}: resultat SQL invalide")
            actual = result[0][0]
            independent = csv_total(question['question'], orders, customers, prices)
            recorded = int(ref['revenue_cents'])
            ok = actual == independent == recorded
            table.append([ref['case_id'], number(actual), number(independent), number(recorded), number(actual - independent), 'OK' if ok else 'ECHEC'])
    widths = [max(len(headers[i]), *(len(row[i]) for row in table)) for i in range(len(headers))]
    separator = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'
    def show(row):
        cells = [value.rjust(widths[i]) if 1 <= i <= 4 else value.ljust(widths[i]) for i, value in enumerate(row)]
        print('| ' + ' | '.join(cells) + ' |')
    print('\nCOMPARAISON DES REFERENCES TD1')
    print('Montants en centimes EUR HT - base SQLite ouverte en lecture seule.\n')
    print(separator)
    show(headers)
    print(separator)
    for row in table:
        show(row)
    print(separator)
    success = sum(row[-1] == 'OK' for row in table)
    print(f'\nBilan : {success}/20 concordances (SQL = CSV = reference).')
    print('SQL : requetes du fichier reference-td1.csv executees sur la base.')
    print('CSV : calcul Python separe sur les trois fichiers sources.')
    return 0 if success == 20 else 1


In [2]:
main()



COMPARAISON DES REFERENCES TD1
Montants en centimes EUR HT - base SQLite ouverte en lecture seule.

+----------+-------------+------------+-----------+---------------+--------+
| Question | SQL execute | Calcul CSV | Reference | Ecart SQL/CSV | Statut |
+----------+-------------+------------+-----------+---------------+--------+
| C01      |     792 800 |    792 800 |   792 800 |             0 | OK     |
| C02      |     590 320 |    590 320 |   590 320 |             0 | OK     |
| C03      |     936 350 |    936 350 |   936 350 |             0 | OK     |
| C04      |   2 826 250 |  2 826 250 | 2 826 250 |             0 | OK     |
| C05      |      38 480 |     38 480 |    38 480 |             0 | OK     |
| C06      |           0 |          0 |         0 |             0 | OK     |
| C07      |      13 970 |     13 970 |    13 970 |             0 | OK     |
| C08      |     605 580 |    605 580 |   605 580 |             0 | OK     |
| C09      |     487 980 |    487 980 |   487 980 | 

## Comparaison des references avec le modele E
Lecture des traces reelles sauvegardees dans `traces_E.CSV`.

In [3]:
import csv
from pathlib import Path
from IPython.display import HTML, display
from html import escape

root = Path.cwd()
with (root / "reference-td1.csv").open(encoding="utf-8", newline="") as f:
    references = {r["case_id"]: int(r["revenue_cents"]) for r in csv.DictReader(f)}
with (root / "traces_E.CSV").open(encoding="utf-8", newline="") as f:
    attempts = list(csv.DictReader(f))
# Retenir la derniere tentative reelle du modele E pour chaque question.
latest = {}
for row in attempts:
    if row["nature"] == "real" and row["provider"] == "openrouter" and row["model_requested"] == "dots-studio/dots-3-note-preview:free":
        latest[row["case_id"]] = row

def amount(value):
    return format(value, ",").replace(",", " ")

labels = {"correct": "Correct", "incorrect": "Incorrect", "sql_invalid": "SQL invalide", "sql_rejected": "SQL refuse"}
body = []
correct = 0
for case, reference in references.items():
    row = latest.get(case)
    value = int(row["value"]) if row and row["value"] != "" else None
    valid = row is not None and row["verdict"] in ("correct", "incorrect")
    matches = valid and value == reference
    correct += matches
    verdict = ("correct" if matches else "incorrect") if valid else row["verdict"] if row else "absent"
    color = "#dcfce7" if matches else "#fee2e2" if value is not None else "#fef3c7"
    values = [case, amount(reference), amount(value) if value is not None else "--", amount(value - reference) if value is not None else "--", labels.get(verdict, verdict), row["attempts"] if row else "--"]
    cells = "".join('<td style="padding:8px 16px;text-align:' + ('right' if i in (1, 2, 3, 5) else 'left') + '">' + escape(str(v)) + '</td>' for i, v in enumerate(values))
    body.append('<tr style="background:' + color + ';color:#172033;border-bottom:1px solid #ddd">' + cells + '</tr>')
headers = ["Question", "Reference HT (ct)", "Modele E HT (ct)", "Ecart modele - ref.", "Verdict", "Tentative"]
head = ''.join('<th style="padding:10px 16px;text-align:left">' + escape(h) + '</th>' for h in headers)
html = '<h3>References / Modele E - Dots</h3><p><b>' + str(correct) + '/' + str(len(references)) + ' correctes (' + format(100 * correct / len(references), '.0f') + '%)</b> - montants en centimes EUR HT</p>'
html += '<table style="border-collapse:collapse;font-family:system-ui"><thead style="background:#1e293b;color:white"><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'
html += '<p>Vert : correct ; rouge : incorrect ; jaune : aucune valeur exploitable. -- signifie une valeur absente, pas zero. Aucun nouvel appel au modele.</p>'
display(HTML(html))


Question,Reference HT (ct),Modele E HT (ct),Ecart modele - ref.,Verdict,Tentative
C01,792 800,792 800,0,Correct,1
C02,590 320,590 320,0,Correct,1
C03,936 350,936 350,0,Correct,1
C04,2 826 250,2 826 250,0,Correct,1
C05,38 480,38 480,0,Correct,1
C06,0,0,0,Correct,1
C07,13 970,13 970,0,Correct,1
C08,605 580,0,-605 580,Incorrect,1
C09,487 980,47 900,-440 080,Incorrect,1
C10,558 980,0,-558 980,Incorrect,1


## Step 3 : syntheses des cinq modeles
Cette cellule consolide les JSONL bruts (toutes les tentatives), remplit les deux CSV de synthese et reverifie les verdicts sans appel aux modeles. Les fichiers individuels restent conserves.

In [4]:
import csv
import json
import runpy
from collections import Counter
from pathlib import Path
from html import escape
from IPython.display import HTML, display

root = Path.cwd()
kit = runpy.run_path(str(root / 'skeleton-multi-llm-en.py'))
config = json.loads((root / 'modeles-autorises.json').read_text(encoding='utf-8'))
models = config['modeles']
questions = json.loads((root / 'questions-td1.json').read_text(encoding='utf-8'))['questions']
refs = kit['load_references'](root / 'reference-td1.csv')
schema = (root / 'schema.sql').read_text(encoding='utf-8')
causes = {
('A','C19'): "birth_date = date('2026-09-25','-25 years') ne retient que le jour du 25e anniversaire, pas tous les clients de 25 ans.",
('B','C02'): "Bornes de naissance 2025-09-25 / 2026-09-25 : selection de nourrissons au lieu de clients de 25 ans.",
('B','C08'): "Naissances filtrees dans [2001-09-24,2002-09-24[ au lieu de ]2000-09-24,2001-09-24].",
('C','C02'): "Alias AS age dans WHERE : syntaxe SQLite invalide ; le terme utilise aussi l'annee de naissance au lieu de l'age.",
('C','C08'): "birth_date +1 year teste la premiere annee de vie, pas le 25e anniversaire ; ajout indu d'une periode de commande le 24 septembre.",
('C','C09'): "Difference des annees comparee a CASE 0/1 au lieu de 25 + correction d'anniversaire.",
('C','C10'): "Difference des annees =25 et anniversaire passe : exclut les clients dont la difference vaut 26 mais qui ont encore 25 ans.",
('C','C11'): "Bornes de naissance 2025/2026 incompatibles avec 25 ans en 2026.",
('C','C17'): "birth_date > reference -26 years ET birth_date <= reference inclut aussi les clients de moins de 25 ans ; borne -25 years absente.",
('C','C19'): "Les modificateurs de date dependent de la difference des annees du client ; ils ne definissent pas l'intervalle fixe ]2000-09-25,2001-09-25].",
('D','C09'): "Utilise o.date au lieu de la reference 2026-09-26 ; mois-jour >= anniversaire ET < meme anniversaire : condition impossible.",
('D','C10'): "Branche difference des annees =24 avant anniversaire : selectionne des clients de 23 ans ; il fallait difference =26 avant anniversaire pour avoir 25 ans.",
('D','C15'): "Mois-jour <=09-25 ET >09-25 : contradiction, car les dates comparees ont le meme mois et jour ; annee de naissance ignoree.",
('D','C16'): "Frontieres anniversaire inversees : < au lieu de <= pour 2001, >= au lieu de > pour 2000 ; le 25 septembre est mal classe.",
('D','C17'): "Frontieres anniversaire inversees : < au lieu de <= pour 2001, >= au lieu de > pour 2000 ; le 25 septembre est mal classe.",
('D','C19'): "Compare strftime('%Y',...) (annee seule) a une date YYYY-MM-DD et ne pose pas la borne du 26e anniversaire : ne selectionne pas exactement 25 ans.",
('E','C08'): "Les comparaisons birth_date || '-00-00-00' <= reference ET birth_date || '-23-59-59' >= reference sont incompatibles avec une naissance 25 ans auparavant.",
('E','C09'): "birth_date = '2001-09-26' ne retient que les clients ayant leur 25e anniversaire ce jour, pas tous ceux de 25 ans.",
('E','C10'): "Mois-jour commande >= anniversaire ET < meme anniversaire (hors 29 fevrier absent des donnees) : filtre contradictoire.",
('E','C11'): "AS birth_date_formatted et virgule dans WHERE : syntaxe invalide, aucun montant exploitable.",
('E','C15'): "Difference des annees =25 et anniversaire passe exclut les clients nes fin 2000 ayant encore 25 ans.",
('E','C17'): "Difference des annees =25 et anniversaire passe exclut les clients nes fin 2000 ayant encore 25 ans.",
('E','C18'): "Difference des annees =25 et anniversaire passe exclut les clients dont la difference vaut 26 mais qui ont encore 25 ans a la commande.",
('E','C19'): "CAST((strftime(...) AS INTEGER) : parentheses mal placees, AS apparait dans une expression au lieu du CAST ; syntaxe invalide.",
}
all_rows, finals, audit = [], [], []
fields = None
for letter, model in zip('ABCDE', models):
    path = root / f'state-{letter}.jsonl'
    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()] if path.exists() else []
    latest = {}
    for row in records:
        assert row['provider'] == model['gate'] and row['model_requested'] == model['identifiant_exact']
        assert row['nature'] == 'real', 'Ne pas melanger les natures dans ce lot reel'
        fields = fields or list(row)
        all_rows.append(row)
        key = row['case_id']
        if key not in latest or (int(row['attempts']), row['date_utc']) >= (int(latest[key]['attempts']), latest[key]['date_utc']):
            latest[key] = row
    for question in questions:
        case = question['id']
        row = latest.get(case)
        final = dict(case_id=case, model_id=model['identifiant_exact'], trace_file=path.name if row else '', cell_id=f"{model['gate']}:{model['identifiant_exact']}:{case}", nature=row['nature'] if row else '', status=row['verdict'] if row else 'not_executed', value_cents=row['value'] if row else '', reference_cents=refs[case], explanation='')
        if row:
            assert str(row['reference']) == str(refs[case])
            expected = kit['build_prompt'](schema, question['question'])[2]
            if row['prompt_sha256'] != expected:
                audit.append(f'{letter}/{case}: empreinte differente du prompt local ; comparer les experiences avant classement.')
            judged = kit['judge'](root / 'data/boutique.sqlite', row['sql'], refs[case])
            if row['verdict'] in ('correct','incorrect','sql_invalid','sql_rejected'):
                assert judged['verdict'] == row['verdict'], (letter, case, judged)
                assert str(judged['value']) == str(row['value']), (letter,case,'value')
            reason = 'Montant conforme a la reference apres execution SQLite.' if row['verdict']=='correct' else causes.get((letter,case), 'Erreur transport/acces : ' + row['error'] if row['error'] else row['verdict_detail'])
            reason += f" Preuve : {path.name}, cell_id={row['cell_id']}, tentative={row['attempts']}; HTTP={row['http_status']}; verdict_detail={row['verdict_detail']}; error={row['error'] or '(vide)'}."
            if row['cost_native'] in ('',None):
                reason += ' Cout natif non fourni : mesure absente, pas zero.'
            if row['prompt_sha256'] != expected:
                reason += ' ATTENTION : empreinte differente du prompt local.'
            final['explanation'] = reason
        else:
            final['explanation'] = 'Trace absente du depot ; execution non attestee.'
        finals.append(final)

def write_csv(name, rows, columns=None):
    with (root / name).open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=columns or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
write_csv('traces-td1.csv', all_rows, fields)
write_csv('model-question-comparison-en.csv', finals)

with (root / 'portefeuille-modeles-2026-09-28.csv').open(encoding='utf-8-sig',newline='') as f:
    portfolio=list(csv.DictReader(f))
summary=[]
for letter, model in zip('ABCDE',models):
    ident=model['identifiant_exact']
    historic=next((p for p in portfolio if p['identifiant_gate']==ident and p['score_public']),None)
    own=[r for r in finals if r['model_id']==ident]
    def rate(family):
        ids={q['id'] for q in questions if ('+' in q['type'] if family=='combined' else '+' not in q['type'] and (q['type']=='period' if family=='period' else q['type'].startswith('age') if family=='age' else q['type']=='product list'))}
        selected=[r for r in own if r['case_id'] in ids]
        return f"{sum(r['status']=='correct' for r in selected)}/{len(selected)} correct; " + '; '.join(f'{k}={v}' for k,v in Counter(r['status'] for r in selected).items())
    paid = {'A':('0.15','0.60'),'B':('0.075','0.30'),'D':('0.80','4.00')}
    free_url='https://openrouter.ai/' + ident
    summary.append(dict(provider=model['gate'],model_id=ident,
        public_index=historic['score_public'] if historic else 'N/A - absent du portefeuille; indice public comparable non documente dans les sources consultees',
        index_date='2026-09-28' if historic else '2026-10-21',
        index_source='portefeuille-modeles-2026-09-28.csv; provenance: portefeuille-modeles-en.md (classement public historique fourni, URL precise non fournie)' if historic else free_url,
        paid_route_same_model='Groq Developer, meme identifiant exact' if letter in paid else 'N/A - page officielle consultee ne documente que la route gratuite de ce modele exact',
        input_usd_per_million=paid[letter][0] if letter in paid else 'N/A',output_usd_per_million=paid[letter][1] if letter in paid else 'N/A',
        price_date='2026-10-21',price_source='https://console.groq.com/docs/models' if letter in paid else free_url,
        access_used='Groq Free selon protocole; 20 traces reelles HTTP 200' if model['gate']=='groq' else 'OpenRouter :free; prix maximum entree/sortie=0; 20 traces reelles HTTP 200',
        period_results=rate('period'),age_results=rate('age'),product_results=rate('product'),combined_results=rate('combined'),
        trace_files=f'state-{letter}.jsonl; ' + ('traces_E.CSV' if letter=='E' else f'traces-{letter}.csv') + '; traces-td1.csv',
        limits='20 cas seulement; 1 tentative/cellule; aucune reprise ni erreur acces dans ce lot; temperature=0; timeout=45s; intervalle minimal=3s; sortie='+str(model['parametres'].get('max_tokens',model['parametres'].get('max_completion_tokens')))+' tokens. Familles exclusives: periode=6, age=5, produit=4, combinees=5. Quotas propres aux comptes, non deduits des limites Developer. Indices historiques non revalides; tarifs consultes apres le lot, pas cout facture. Hypothese de routage limitee a ce lot, pas classement general.'))
write_csv('five-model-summary-en.csv',summary)
print(f'{len(all_rows)} tentatives conservees ; {len(finals)} resultats finaux ; {len(summary)} modeles.')
for s in summary:
    own=[r for r in finals if r['model_id']==s['model_id']]
    print(s['model_id'],sum(r['status']=='correct' for r in own),'/20',s['period_results'],s['age_results'],s['product_results'],s['combined_results'])
print('Audit empreintes :', '\n'.join(audit) if audit else 'identiques au prompt local pour les 100 cellules.')
print('Colonnes manquantes dans traces-B.csv : donnees completes conservees dans state-B.jsonl et traces-td1.csv.')
print('Couts natifs absents :',sum(r['cost_native'] in ('',None) for r in all_rows),'(laisser vides).')


100 tentatives conservees ; 100 resultats finaux ; 5 modeles.
openai/gpt-oss-120b 19 /20 6/6 correct; correct=6 5/5 correct; correct=5 4/4 correct; correct=4 4/5 correct; correct=4; incorrect=1
openai/gpt-oss-20b 18 /20 6/6 correct; correct=6 3/5 correct; incorrect=2; correct=3 4/4 correct; correct=4 5/5 correct; correct=5
cohere/north-mini-code:free 13 /20 6/6 correct; correct=6 0/5 correct; sql_invalid=1; incorrect=4 4/4 correct; correct=4 3/5 correct; correct=3; incorrect=2
qwen/qwen3.8-27b 14 /20 6/6 correct; correct=6 3/5 correct; correct=3; incorrect=2 4/4 correct; correct=4 1/5 correct; incorrect=4; correct=1
dots-studio/dots-3-note-preview:free 12 /20 6/6 correct; correct=6 1/5 correct; correct=1; incorrect=3; sql_invalid=1 4/4 correct; correct=4 1/5 correct; incorrect=3; correct=1; sql_invalid=1
Audit empreintes : identiques au prompt local pour les 100 cellules.
Colonnes manquantes dans traces-B.csv : donnees completes conservees dans state-B.jsonl et traces-td1.csv.
Couts na

In [5]:
import csv
from pathlib import Path
from html import escape
from IPython.display import HTML, display

with Path("model-question-comparison-en.csv").open(encoding="utf-8", newline="") as f:
    comparison = list(csv.DictReader(f))
models_order = list(dict.fromkeys(r["model_id"] for r in comparison))
cases = sorted({r["case_id"] for r in comparison})
lookup = {(r["model_id"], r["case_id"]): r for r in comparison}
colors = {"correct": "#dcfce7", "incorrect": "#fee2e2", "sql_invalid": "#fef3c7", "not_executed": "#e2e8f0"}
symbols = {"correct": "OK", "incorrect": "ERR", "sql_invalid": "SQL", "not_executed": "--"}
head = "<th>Modele</th>" + "".join("<th>" + c + "</th>" for c in cases) + "<th>Total</th>"
body = []
for model in models_order:
    cells = []
    success = 0
    for case in cases:
        row = lookup[(model, case)]
        success += row["status"] == "correct"
        title = "Valeur=" + (row["value_cents"] or "absente") + "; reference=" + row["reference_cents"] + "; " + row["explanation"]
        cells.append('<td title="' + escape(title, quote=True) + '" style="background:' + colors.get(row["status"], "#fed7aa") + ';color:#172033;padding:8px;text-align:center">' + symbols.get(row["status"], row["status"]) + '</td>')
    body.append('<tr><th style="text-align:left;padding:8px">' + escape(model) + '</th>' + ''.join(cells) + '<th>' + str(success) + '/20</th></tr>')
display(HTML('<h3>Matrice modeles x questions</h3><p>Vue calculee depuis model-question-comparison-en.csv. Survoler une cellule pour voir les montants et la cause.</p><div style="overflow-x:auto"><table style="border-collapse:collapse;font-family:system-ui"><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div><p>OK : correct ; ERR : montant incorrect ; SQL : requete invalide ; -- : non execute. Chaque cellule represente la derniere tentative reelle.</p>'))


Matrice modeles x questions Vue calculee depuis model-question-comparison-en.csv. Survoler une cellule pour voir les montants et la cause. Modele C01 C02 C03 C04 C05 C06 C07 C08 C09 C10 C11 C12 C13 C14 C15 C16 C17 C18 C19 C20 Total openai/gpt-oss-120b OK OK OK OK OK OK OK OK OK OK OK OK OK OK OK OK OK OK ERR OK 19/20 openai/gpt-oss-20b OK ERR OK OK OK OK OK ERR OK OK OK OK OK OK OK OK OK OK OK OK 18/20 cohere/north-mini-code:free OK SQL OK OK OK OK OK ERR ERR ERR ERR OK OK OK OK OK ERR OK ERR OK 13/20 qwen/qwen3.8-27b OK OK OK OK OK OK OK OK ERR ERR OK OK OK OK ERR ERR ERR OK ERR OK 14/20 dots-studio/dots-3-note-preview:free OK OK OK OK OK OK OK ERR ERR ERR SQL OK OK OK ERR OK ERR ERR SQL OK 12/20 OK : correct ; ERR : montant incorrect ; SQL : requete invalide ; -- : non execute. Chaque cellule represente la derniere tentative reelle.

## Step 4 : support de pleniere

Le tableau ci-dessous compte un echec SQL comme une question manquee, tout en le distinguant d'un montant incorrect. Il est calcule a partir du CSV de synthese.


In [6]:
import csv
from collections import defaultdict, Counter
from html import escape
from IPython.display import HTML, display

with open("model-question-comparison-en.csv", encoding="utf-8", newline="") as f:
    plenary_rows = list(csv.DictReader(f))
by_case = defaultdict(list)
for row in plenary_rows:
    by_case[row["case_id"]].append(row)
conventions = {
    "C08": "Age revolu au 24/09/2026 : mois et jour, toutes les commandes des clients retenus.",
    "C09": "Age au 26/09/2026, pas age a la commande ni anniversaire du jour uniquement.",
    "C10": "25 ans a chaque commande : inclure les clients dont le 26e anniversaire est encore a venir.",
    "C17": "Cumuler periode [debut,fin[, produits a/b/c et age revolu a la reference.",
    "C19": "Cumuler septembre [01/09,01/10[, produit d et age revolu au 25/09/2026 ; syntaxe SQL valide.",
}
html_rows = []
for case, items in sorted(by_case.items(), key=lambda pair: (-sum(r["status"] != "correct" for r in pair[1]), pair[0])):
    missed = sum(r["status"] != "correct" for r in items)
    if missed < 3:
        continue
    invalid = sum(r["status"] == "sql_invalid" for r in items)
    html_rows.append("<tr>" + "".join("<td style='padding:8px;border:1px solid #aaa'>" + escape(str(v)) + "</td>" for v in [case, f"{missed}/5 dont {invalid} SQL invalide(s)", items[0]["reference_cents"], conventions[case]]) + "</tr>")
display(HTML("<h3>Questions manquees par la majorite des modeles</h3><table style='border-collapse:collapse'><tr><th>Question</th><th>Echecs</th><th>Reference HT (ct)</th><th>Convention a expliquer</th></tr>" + "".join(html_rows) + "</table>"))
print("Accords partiels sur une valeur fausse (pas necessairement une majorite) :")
for case, items in sorted(by_case.items()):
    counts = Counter(r["value_cents"] for r in items if r["status"] == "incorrect" and r["value_cents"] != "")
    for value, count in counts.items():
        if count >= 2:
            print(f"  {case} : {count} modeles repondent {value}, reference {items[0]['reference_cents']}.")
print("Ces accords partiels ne prouvent pas un consensus majoritaire faux. Le vote reste a etudier en TD2.")


Question,Echecs,Reference HT (ct),Convention a expliquer
C19,4/5 dont 1 SQL invalide(s),1990,"Cumuler septembre [01/09,01/10[, produit d et age revolu au 25/09/2026 ; syntaxe SQL valide."
C08,3/5 dont 0 SQL invalide(s),605580,"Age revolu au 24/09/2026 : mois et jour, toutes les commandes des clients retenus."
C09,3/5 dont 0 SQL invalide(s),487980,"Age au 26/09/2026, pas age a la commande ni anniversaire du jour uniquement."
C10,3/5 dont 0 SQL invalide(s),558980,25 ans a chaque commande : inclure les clients dont le 26e anniversaire est encore a venir.
C17,3/5 dont 0 SQL invalide(s),258180,"Cumuler periode [debut,fin[, produits a/b/c et age revolu a la reference."


Accords partiels sur une valeur fausse (pas necessairement une majorite) :
  C08 : 2 modeles repondent 0, reference 605580.
  C09 : 2 modeles repondent 0, reference 487980.
  C10 : 2 modeles repondent 119410, reference 558980.
Ces accords partiels ne prouvent pas un consensus majoritaire faux. Le vote reste a etudier en TD2.
